In [ ]:
import torch
import pandas as pd
import geopandas as gpd
import ast
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
# from google.colab import drive
# drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the file
path = "../data/latents_2_92160_for_intervention.pt"
data = torch.load(path, map_location=torch.device('cpu'))

# 2. Extract components
saved_latents = data["latents"]
# Assuming meta was saved as a tensor of [lat, lon]
saved_meta = data["meta"]
# 3. Print Summary Stats
print("--- 📂 File Inspection: latents.pt ---")
print(f"Total Samples:    {saved_latents.shape[0]}")
print(f"Latent Shape:     {saved_latents.shape[1]}")

# 4. Detailed Metric Breakdown (First 5 Samples)
print("\n--- 🛰️ Extracted Metrics Per Sample ---")
header = f"{'Idx':<5} | {'Lat':<10} | {'Lon':<10} | {'Cloud':<8} | {'Mean Elev':<10} | {'Std Elev':<10} | {'Range Elev':<10} |{'Time angle':<10} | Latent Head"
print(header)
print("-" * len(header))

for i in range(min(5, len(saved_latents))):
    # If you saved extra metrics in 'meta' or a separate dict, adjust indexing here:
    lat, lon, cloud_score, mean_elev, std_elev, range_elev, time = saved_meta[i].tolist()

    # Example of how to pull the first few values of the latent for a "fingerprint"
    # Fix: Flatten the tensor before taking the first 3 elements to get a flat list of numbers
    fingerprint = saved_latents[i].flatten()[:3].tolist()
    fp_str = ", ".join([f"{x:.3f}" for x in fingerprint])

    print(f"{i:<5} | {lat:<10.4f} | {lon:<10.4f} | {cloud_score:<8.4f} | {mean_elev:<10.2f} | {std_elev:<10.2f} | {range_elev:<10.2f} | {time:<10.2f} | {fp_str}")

# 5. Numerical Health Check
print("\n--- 🔍 Numerical Health Check ---")
lat_std = saved_latents.std(dim=0).mean().item()
print(f"Average Latent Std Dev: {lat_std:.6f}")

if lat_std < 1e-6:
    print("⚠️  CRITICAL: Latent vectors are identical across samples. Check model input/normalization.")
elif torch.isnan(saved_latents).any():
    print("❌ ERROR: NaNs detected in the saved tensors.")
else:
    print("✅ Latents appear diverse and healthy.")

In [ ]:
# Define a simple class to hold the model parameters
class HModel:
    def __init__(self, rotation_matrix, translation_vec, lambda_vec):
        # Convert lists to PyTorch tensors
        self.rotation_matrix = torch.tensor(rotation_matrix, dtype=torch.float32).to(device)
        self.t_vec = torch.tensor(translation_vec, dtype=torch.float32).to(device)
        self.lambda_vec = torch.tensor(lambda_vec, dtype=torch.float32).to(device)


In [ ]:
import torch
import numpy as np
import pandas as pd
import geopandas as gpd
import ast
import matplotlib.pyplot as plt
from libpysal.weights import KNN
from esda.moran import Moran

def analyze_spatial_autocorrelation(var_title, csv_path, meta_idx, latents_indices=[2, 4, 6, 8, 10, 12], device='cpu'):
    results = []

    # Load geometry data once
    df_geom = pd.read_csv(csv_path)
    columns_to_convert = ['rotation_matrix', 'translation_vec', 'lambda_vec']
    for col in columns_to_convert:
        df_geom[col] = df_geom[col].apply(ast.literal_eval)

    for i in latents_indices:
        print(i)
        path = f"../data/latents_{i}_92160_for_intervention.pt"
        data = torch.load(path, map_location=device)
        X = data["latents"]
        labels = data["meta"][:, meta_idx].cpu().numpy().flatten()

        row_scores = {"Iteration": i}

        for mode in ['Model', 'Baseline']:
            if mode == 'Model':
                # Map iteration index to CSV row (adjust logic if i//6 is not your intended mapping)
                geom_row = df_geom.iloc[i // 2 - 1]
                R = torch.tensor(geom_row['rotation_matrix'], device=device, dtype=torch.float32)
                t = torch.tensor(geom_row['translation_vec'], device=device, dtype=torch.float32)
                l = torch.tensor(geom_row['lambda_vec'], device=device, dtype=torch.float32)
            else:
                # Baseline: Random Rotation, Zero T, Zero Lambda
                N = X.shape[1]
                H = np.random.randn(N, N)
                Q, _ = np.linalg.qr(H)
                if np.linalg.det(Q) < 0: Q[:, 0] = -Q[:, 0]

                R = torch.tensor(Q, device=device, dtype=torch.float32)
                t = torch.zeros(N, device=device)
                l = torch.zeros(N, device=device)

            # --- Transformation ---
            with torch.no_grad():
                X_centered = X.to(device) + t
                X_rotated = torch.matmul(X_centered, R)

                # Identify dims: Use lambda for model, [0,1] for baseline
                if mode == 'Model':
                    top_2 = torch.argsort(l.abs(), descending=True)[:2]
                else:
                    top_2 = torch.tensor([0, 1], device=device)

                Ls = X_rotated[:, top_2].cpu().numpy()
                scales = l[top_2].cpu().numpy()


            # --- Moran's I ---
            gdf = gpd.GeoDataFrame({'label': labels}, geometry=gpd.points_from_xy(Ls[:, 0], Ls[:, 1]))
            w = KNN.from_dataframe(gdf, k=8)
            w.transform = 'R'
            mi = Moran(gdf['label'], w)

            row_scores[f"{mode}_Moran"] = round(mi.I, 4)

        results.append(row_scores)

    # --- Generate Tables ---
    summary_df = pd.DataFrame(results)
    print("\n--- Spatial Autocorrelation Summary ---")
    print(summary_df.to_string(index=False))

    # LaTeX Generation
    latex_table = summary_df.to_latex(index=False,
                                     caption=f"Spatial Autocorrelation (Moran's I) for {var_title}",
                                     label=f"tab:moran_{var_title.lower()}",
                                     column_format="|c|c|c|c|c|")
    return latex_table

# Usage:
latex_code = analyze_spatial_autocorrelation("Latitude", "../Data/H_geometry_results.csv", 3)
print(latex_code)

In [ ]:
latex_code = analyze_spatial_autocorrelation("Latitude", "../Data/L_geometry_results.csv", 0)
print(latex_code)

In [ ]:
import ast
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from esda.moran import Moran
from libpysal.weights import KNN
from scipy.optimize import minimize


# --- Circular Statistics Helpers ---
def circular_mean(angles):
    """Computes the directional/circular mean of an array of angles (in radians)."""
    return np.arctan2(np.sum(np.sin(angles)), np.sum(np.cos(angles)))


def circular_correlation(alpha, beta):
    """Computes the Jammalamadaka–Sarma circular correlation coefficient (radians)."""
    mu_alpha = circular_mean(alpha)
    mu_beta = circular_mean(beta)

    numerator = np.sum(np.sin(alpha - mu_alpha) * np.sin(beta - mu_beta))
    denominator = np.sqrt(
        np.sum(np.sin(alpha - mu_alpha) ** 2)
        * np.sum(np.sin(beta - mu_beta) ** 2)
    )

    return numerator / denominator if denominator != 0 else 0.0


def mean_resultant_length(phi, theta):
    """
    Mean resultant length of angular differences delta = theta - phi.
    R in [0, 1].  R ≈ 1 means the labels align perfectly with spatial angles
    around the chosen center (up to a global rotation).
    """
    delta = theta - phi
    C = np.mean(np.cos(delta))
    S = np.mean(np.sin(delta))
    return np.sqrt(C**2 + S**2)


def neg_abs_circ_corr(center, points, labels):
    """
    Objective for the optimizer: negative |circular correlation|
    between spatial angles around 'center' and the labels.
    """
    dx = points[:, 0] - center[0]
    dy = points[:, 1] - center[1]
    # Guard against the optimizer landing exactly on a data point
    if np.any((np.abs(dx) < 1e-12) & (np.abs(dy) < 1e-12)):
        return 0.0
    phi = np.arctan2(dy, dx)
    corr = circular_correlation(phi, labels)
    if np.isnan(corr):
        return 0.0
    return -abs(corr)


def find_optimal_center(points, labels, init_guesses=None, method="Nelder-Mead"):
    """
    Find the center (a, b) that maximizes |circular_correlation(phi, labels)|.

    Parameters
    ----------
    points : (n, 2) ndarray
    labels : (n,) ndarray  — angles in radians
    init_guesses : list of (x, y) tuples/arrays.  If None, uses mean and bbox centers.
    method : str — optimizer passed to scipy.optimize.minimize

    Returns
    -------
    best_center : (2,) ndarray
    best_corr   : float  — the maximized |correlation|
    best_mrl    : float  — mean resultant length at that center
    """
    if init_guesses is None:
        mean_c = np.mean(points, axis=0)
        bbox_c = (np.min(points, axis=0) + np.max(points, axis=0)) / 2.0
        init_guesses = [mean_c, bbox_c]

    best_center = None
    best_corr = -1.0
    best_mrl = 0.0

    for guess in init_guesses:
        result = minimize(
            neg_abs_circ_corr,
            guess,
            args=(points, labels),
            method=method,
            options={"xatol": 1e-7, "fatol": 1e-7, "maxiter": 500},
        )
        corr = -result.fun
        if corr > best_corr:
            best_corr = corr
            best_center = result.x

    # Compute MRL at the winning center
    phi_opt = np.arctan2(points[:, 1] - best_center[1], points[:, 0] - best_center[0])
    best_mrl = mean_resultant_length(phi_opt, labels)

    return best_center, best_corr, best_mrl


# --- Main Analysis Function ---
def analyze_spatial_autocorrelation(
    var_title,
    csv_path,
    meta_idx,
    latents_indices=[2, 4, 6, 8, 10, 12],
    device="cpu",
    circular=False,
    n_baseline_runs=4,
    time=False,
):
    results = []

    # Load geometry data once
    df_geom = pd.read_csv(csv_path)
    columns_to_convert = ["rotation_matrix", "translation_vec", "lambda_vec"]
    for col in columns_to_convert:
        df_geom[col] = df_geom[col].apply(ast.literal_eval)

    for i in latents_indices:
        print(i)
        path = f"../data/latents_{i}_92160_for_intervention.pt"
        data = torch.load(path, map_location=device)
        if time:
            current_meta = data["meta"]  # Use meta from current data load
            lats = current_meta[:, 0]
            alts = current_meta[:, 3]
            mask = (lats > 15) & (lats < 66) & (alts < 4000)
            X = data["latents"][mask]  # Apply mask to current latents
            labels = (
                current_meta[:, -1][mask].cpu().numpy()
            )  # Apply mask to current meta and convert to numpy
        else:
            X = data["latents"]
            labels = data["meta"][:, meta_idx].cpu().numpy().flatten() * np.pi / 180

        row_scores = {"Iteration": i}

        # ==========================================
        # 1. MODEL TRANSFORMATION
        # ==========================================
        geom_row = df_geom.iloc[0]
        R_m = torch.tensor(
            geom_row["rotation_matrix"], device=device, dtype=torch.float32
        )
        t_m = torch.tensor(
            geom_row["translation_vec"], device=device, dtype=torch.float32
        )
        l_m = torch.tensor(
            geom_row["lambda_vec"], device=device, dtype=torch.float32
        )

        with torch.no_grad():
            X_centered_m = X.to(device) + t_m
            X_rotated_m = torch.matmul(X_centered_m, R_m)
            top_2_m = torch.argsort(l_m.abs(), descending=True)[:2]

            Ls_model = X_rotated_m[:, top_2_m].cpu().numpy()

        # Calculate Model Moran's I
        gdf_m = gpd.GeoDataFrame(
            {"label": labels},
            geometry=gpd.points_from_xy(Ls_model[:, 0], Ls_model[:, 1]),
        )
        w_m = KNN.from_dataframe(gdf_m, k=8)  # TODO explain this k in report
        w_m.transform = "R"
        row_scores["Model_Moran"] = round(Moran(gdf_m["label"], w_m).I, 4)
        # TODO calulate confidence intervals for Moran's I

        # Calculate Model Circular Correlation (optimized center)
        if circular:
            center_m_opt, corr_m_opt, mrl_m_opt = find_optimal_center(
                Ls_model, labels
            )

            # Also compute heuristics for reference / plotting
            center_m_mean = np.mean(Ls_model, axis=0)
            center_m_bbox = (np.min(Ls_model, axis=0) + np.max(Ls_model, axis=0)) / 2.0

            row_scores["Model_CircCorr"] = round(corr_m_opt, 4)
            row_scores["Model_MRL"] = round(mrl_m_opt, 4)

        # ==========================================
        # 2. BASELINE TRANSFORMATION (Averaged over N runs)
        # ==========================================
        baseline_morans = []
        baseline_circcorrs = []
        baseline_mrls = []
        N_dims = X.shape[1]

        for _ in range(n_baseline_runs):
            # Random Rotation
            H = np.random.randn(N_dims, N_dims)
            Q, _ = np.linalg.qr(H)
            if np.linalg.det(Q) < 0:
                Q[:, 0] = -Q[:, 0]

            R_b = torch.tensor(Q, device=device, dtype=torch.float32)
            t_b = torch.zeros(N_dims, device=device)

            with torch.no_grad():
                X_centered_b = X.to(device) + t_b
                X_rotated_b = torch.matmul(X_centered_b, R_b)
                top_2_b = torch.tensor([0, 1], device=device)

                Ls_base = X_rotated_b[:, top_2_b].cpu().numpy()

            # Baseline Moran's I step
            gdf_b = gpd.GeoDataFrame(
                {"label": labels},
                geometry=gpd.points_from_xy(Ls_base[:, 0], Ls_base[:, 1]),
            )
            w_b = KNN.from_dataframe(gdf_b, k=8)
            w_b.transform = "R"
            baseline_morans.append(Moran(gdf_b["label"], w_b).I)

            # Baseline Circular Correlation step (also optimized)
            if circular:
                center_b_opt, corr_b_opt, mrl_b_opt = find_optimal_center(
                    Ls_base, labels
                )
                baseline_circcorrs.append(corr_b_opt)
                baseline_mrls.append(mrl_b_opt)

        # Average baseline scores
        row_scores["Baseline_Moran"] = round(np.mean(baseline_morans), 4)
        if circular:
            row_scores["Baseline_CircCorr"] = round(np.mean(baseline_circcorrs), 4)
            row_scores["Baseline_MRL"] = round(np.mean(baseline_mrls), 4)

        # ==========================================
        # 3. HEATMAP GENERATION
        # ==========================================
        cmap_choice = "twilight" if circular else "viridis"

        plt.figure(figsize=(6, 5))
        plt.scatter(
            Ls_model[:, 0],
            Ls_model[:, 1],
            c=labels,
            cmap=cmap_choice,
            s=5,
            alpha=0.5,
        )
        plt.colorbar(label=var_title)

        if circular:
            # Plot OPTIMAL center (the one that maximizes correlation)
            plt.scatter(
                center_m_opt[0],
                center_m_opt[1],
                color="lime",
                marker="*",
                s=250,
                edgecolor="black",
                label=f"Optimal Center (r={round(corr_m_opt, 2)}, R={round(mrl_m_opt, 2)})",
                zorder=4,
            )
            # Plot arithmetic mean center (for reference)
            plt.scatter(
                center_m_mean[0],
                center_m_mean[1],
                color="red",
                marker="X",
                s=150,
                edgecolor="black",
                label="Mean Center",
                zorder=3,
            )
            # Plot bounding box center (for reference)
            plt.scatter(
                center_m_bbox[0],
                center_m_bbox[1],
                color="cyan",
                marker="o",
                s=100,
                facecolors="none",
                edgecolors="cyan",
                linewidths=2,
                label="BBox Center",
                zorder=2,
            )
            plt.legend(loc="best")

        plt.title(
            f"Model Space (Iter {i}) — {'Cyclic' if circular else 'Linear'} Colors"
        )
        plt.xlabel("Latent Dim 1")
        plt.ylabel("Latent Dim 2")
        plt.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()

        results.append(row_scores)

    # --- Generate Tables ---
    summary_df = pd.DataFrame(results)
    print("\n--- Spatial Autocorrelation Summary ---")
    print(summary_df.to_string(index=False))

    # Dynamic column format matches the final column count
    col_fmt = "|" + "c|" * len(summary_df.columns)

    # LaTeX Generation
    latex_table = summary_df.to_latex(
        index=False,
        caption=f"Spatial Autocorrelation Metrics for {var_title}",
        label=f"tab:spatial_{var_title.lower()}",
        column_format=col_fmt,
    )
    return latex_table

In [ ]:
latex_code = analyze_spatial_autocorrelation("Year progression", "../Data/T_geometry_results.csv", meta_idx=-1, circular=True, time = True)
print(latex_code)

In [ ]:
latex_code = analyze_spatial_autocorrelation("Year progression", "T_10_geometry_results.csv", meta_idx=-1, circular=True, time = True, latents_indices=[10])
print(latex_code)

In [ ]:
latex_code = analyze_spatial_autocorrelation("Longitude", "Long_geometry_results.csv", meta_idx=1, circular=True)
print(latex_code)